# Project8 v05 experiment campaign — 2026-10-10
12 training configurations, identical 2,500-image val. Native end-to-end and class-aware extra NMS 0.6 are measured separately. Full visual annotation review remains incomplete. No test images used. Verified archives per trial.


In [ ]:
# 1. Ba pilot don gian: moi luot 2 epoch, cung BEST v0.5
START_TRAINING = False
ALLOW_UNREVIEWED_PILOT = True  # user authorized exploratory pilots; full review remains incomplete
ARCHIVE_INPUT = ''
DATASET_INPUT = ''
CHECKPOINT_INPUT = ''  # search attached notebook output by SHA256
GPU_IDS = '0,1'
EPOCHS = 2
PATIENCE = 0
IMGSZ = 1280
BATCH = 32
CLOSE_MOSAIC = 0
SMOKE_ONLY = False
RUN_ONE_EPOCH_SMOKE = False
OUTPUT_ROOT = '/kaggle/working/BTL-AI-Camera-v05-campaign-20261010'
OLD_NAMES = ['person', 'table', 'chair', 'laptop', 'cell phone', 'backpack', 'book', 'cup']
NAMES = OLD_NAMES[:-1] + ['bottle']
EXPECTED_IMAGES = {'train': 12000, 'val': 2500, 'test': 500}
EXPECTED_BOXES = {'train': 33272, 'val': 8359, 'test': 1645}
BEST_SHA256 = '92adfeb2516861a6958e8adc4a15cf764faa3956078da1ebc81db1fbb7014e31'
MANIFEST_SHA256 = '22a0140ab5e4d454d7e0c46ab24145c6d52d8b949db67ab8142fdd3bb8bb9ccd'
CHECKSUMS_SHA256 = '53a664318ad67bb64f6c894c21e94a6543382b5b9cf859e5ca9d4e283db4a77e'
PORTABLE_YAML_SHA256 = '4b26114dfaf683ce88b3b159cdec3f37f30f9b8168fd260c6cc1a977d718a122'
BASE_AUG = dict(mosaic=0.5, scale=0.5, translate=0.1, fliplr=0.5,
                hsv_h=0.015, hsv_s=0.7, hsv_v=0.4)
LIGHT_AUG = dict(mosaic=0.2, scale=0.25, translate=0.05, fliplr=0.25,
                 hsv_h=0.0075, hsv_s=0.35, hsv_v=0.2)
EXPERIMENTS = [dict(label='L0A0_control', lr0=0.001, aug=BASE_AUG),
               dict(label='L1A0_half_lr', lr0=0.0005, aug=BASE_AUG),
               dict(label='L0A1_light_aug', lr0=0.001, aug=LIGHT_AUG)]
# Shared short-run schedule: flat post-warmup LR, warmup 0.25 epoch, bias LR 0.
# This control is a new 2-epoch baseline; historical 5-epoch schedule differs.


In [ ]:
# 2. Giữ phiên bản Ultralytics đã dùng tạo checkpoint; không cài requirements CPU local
import subprocess, sys, torch
before_torch = (torch.__version__, torch.version.cuda)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.165'], check=True)
import ultralytics, yaml
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from pathlib import Path
import csv, json, hashlib, math, shutil, datetime, gc, zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
assert ultralytics.__version__ == '8.4.165'
assert before_torch == (torch.__version__, torch.version.cuda), 'Torch/CUDA bị thay đổi; cần reset môi trường.'
print('Ultralytics:', ultralytics.__version__, '| Torch/CUDA:', before_torch)
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 3. Dataset archive giu nguyen; checkpoint train la BEST v0.5 theo SHA256
INPUT_ROOT = Path('/kaggle/input')
def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def safe_path(base, relative):
    path = Path(relative)
    if path.is_absolute() or '..' in path.parts:
        raise ValueError(f'Đường dẫn không an toàn: {relative}')
    result = (base / path).resolve()
    if not result.is_relative_to(base.resolve()):
        raise ValueError(f'Đường dẫn ra ngoài dataset: {relative}')
    return result

# Một file Input; giữ đuôi .zip.bin để Kaggle không tự bung lúc upload.
import stat, time
ARCHIVE_SHA256 = '236972ca3e6142d977a29068cf217dc5312c8fb870c8094a98c4960386a0eb5b'
ARCHIVE_NAME = 'project8_v05_trainval_best_v04_20261008.zip.bin'
ARCHIVE_ROOT = None
archive_paths = [Path(ARCHIVE_INPUT)] if ARCHIVE_INPUT else sorted(INPUT_ROOT.rglob(ARCHIVE_NAME))
if len(archive_paths) > 1:
    raise RuntimeError('Có nhiều archive; đặt ARCHIVE_INPUT cho một file cụ thể.')
if archive_paths:
    archive_path = archive_paths[0]
    print('ARCHIVE_HASH_START:', archive_path, flush=True)
    started = time.perf_counter()
    assert file_sha(archive_path) == ARCHIVE_SHA256, 'Sai SHA256 archive train/val + BEST v0.4'
    ARCHIVE_ROOT = Path('/tmp/project8_v05_15k_runtime') / ('archive_' + ARCHIVE_SHA256[:16])
    ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        infos = archive.infolist()
        member_names = [info.filename for info in infos]
        assert len(member_names) == len(set(member_names)) == 29005, 'Archive thiếu/trùng file'
        manifest_bytes = archive.read('project8_v05/manifest.csv')
        assert hashlib.sha256(manifest_bytes).hexdigest() == MANIFEST_SHA256
        archive_rows = list(csv.DictReader(manifest_bytes.decode('utf-8-sig').splitlines()))
        assert len(archive_rows) == 15000
        expected_members = {
            'project8_v05/' + row[key]
            for row in archive_rows if row['split'] in {'train', 'val'}
            for key in ('image_path', 'label_path')
        } | {
            'project8_v05/manifest.csv', 'project8_v05/checksums.json',
            'project8_v05/data.yaml', 'project8_v05/training_readiness.json',
            'BEST_Project8_v04_20261008.pt',
        }
        assert set(member_names) == expected_members, 'Archive có file dư/thiếu hoặc test media'
        assert shutil.disk_usage(ARCHIVE_ROOT).free > sum(info.file_size for info in infos) + 512*1024**2, 'Thiếu ổ tạm để giải nén'
        # Kiểm toàn bộ đường dẫn trước khi ghi bất kỳ file nào.
        for info in infos:
            assert '\\' not in info.filename and not info.is_dir()
            assert not stat.S_ISLNK(info.external_attr >> 16), 'Không nhận symlink'
            safe_path(ARCHIVE_ROOT, info.filename)
        print('ARCHIVE_HASH_PASS: 29005 files; extract locally', flush=True)
        for index, info in enumerate(infos, 1):
            destination = safe_path(ARCHIVE_ROOT, info.filename)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as read, destination.open('wb') as write:
                shutil.copyfileobj(read, write, length=1024*1024)
            assert destination.stat().st_size == info.file_size
            if index % 2000 == 0 or index == len(infos):
                print('UNZIP:', index, '/', len(infos), '| seconds:', round(time.perf_counter()-started, 1), flush=True)
    INPUT_ROOT = ARCHIVE_ROOT
    print('ARCHIVE_READY:', INPUT_ROOT, '| seconds:', round(time.perf_counter()-started, 1), flush=True)
else:
    print('Không có archive; dùng Input đã bung sẵn.', flush=True)

manifests = [Path(DATASET_INPUT) / 'manifest.csv'] if DATASET_INPUT else sorted(INPUT_ROOT.rglob('manifest.csv'))
valid_roots = [p.parent for p in manifests if p.is_file() and file_sha(p) == MANIFEST_SHA256 and (p.parent/'images/train').is_dir()]
if not valid_roots:
    raise FileNotFoundError('Chưa gắn đúng dataset v0.5 15k. Add Input dataset một file .zip.bin hoặc dataset v0.5 đã bung sẵn.')
if len(valid_roots) != 1:
    raise RuntimeError('Có nhiều bản dataset 15k; đặt DATASET_INPUT cho một bản cụ thể.')
SOURCE_DATA = valid_roots[0]
assert file_sha(SOURCE_DATA / 'checksums.json') == CHECKSUMS_SHA256
checksums = json.loads((SOURCE_DATA / 'checksums.json').read_text(encoding='utf-8'))
with (SOURCE_DATA / 'manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
    rows = list(csv.DictReader(stream))
assert len(rows) == 15000 and Counter(r['split'] for r in rows) == EXPECTED_IMAGES
cfg = yaml.safe_load((SOURCE_DATA/'data.yaml').read_text(encoding='utf-8'))
names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][i] for i in range(8)]
assert names == NAMES and cfg['nc'] == 8
assert len({r['pixel_sha256'] for r in rows}) == 15000
expected_files = {r[k] for r in rows for k in ('image_path','label_path')}
assert set(checksums) == expected_files | {'manifest.csv','data.yaml'} and len(checksums) == 30002
assert checksums['manifest.csv'] == MANIFEST_SHA256
# Export YAML is portable; source data.yaml digest remains in the historical release checksums.
assert file_sha(SOURCE_DATA/'data.yaml') == PORTABLE_YAML_SHA256
assert cfg.get('train')=='images/train' and cfg.get('val')=='images/val' and not cfg.get('test')

weight_paths = [Path(CHECKPOINT_INPUT)] if CHECKPOINT_INPUT else sorted(set(Path('/kaggle/input').rglob('*.pt')) | set(INPUT_ROOT.rglob('*.pt')))
matched_weights = [p for p in weight_paths if p.is_file() and file_sha(p) == BEST_SHA256]
if not matched_weights:
    raise FileNotFoundError('Thieu BEST v0.5 dung SHA256; Add Input output notebook v0.5.')
initial = matched_weights[0]  # các bản còn lại, nếu có, đều cùng byte/hash
model_check = YOLO(str(initial))
assert model_check.task == 'detect' and [model_check.names[i] for i in range(8)] == NAMES
print('Dataset:', SOURCE_DATA, '| split:', dict(Counter(r['split'] for r in rows)))
print('BEST v0.5:', initial, '| SHA256:', file_sha(initial))
print('ID 7 bottle gồm cup+bottle; giữ nguyên 8 ID từ best v0.4.')
del model_check; gc.collect()
readiness = json.loads((SOURCE_DATA/'training_readiness.json').read_text(encoding='utf-8'))
assert readiness['manifest_sha256'] == MANIFEST_SHA256
TRAIN_GATE = (readiness.get('training_ready') is True and readiness.get('full_visual_review_complete') is True and all(r['review_stage']=='full_review_complete' for r in rows))
print('Full visual review gate:', TRAIN_GATE)

assert {split:sum(int(r['boxes']) for r in rows if r['split']==split) for split in EXPECTED_IMAGES} == EXPECTED_BOXES


In [ ]:
# 4. Stage và kiểm SHA256/tọa độ toàn bộ train+val; test không vào thư mục train
WORK = Path('/tmp/project8_v05_15k_runtime')
DATA_DIR = SOURCE_DATA if ARCHIVE_ROOT is not None else WORK / 'dataset'
DATA_DIR.mkdir(parents=True, exist_ok=True)
train_val = [r for r in rows if r['split'] in {'train','val'}]

def stage_record(row):
    for key, hash_key in [('image_path','image_sha256'),('label_path','label_sha256')]:
        src = safe_path(SOURCE_DATA, row[key]); dst = safe_path(DATA_DIR, row[key])
        assert checksums[row[key]] == row[hash_key]
        assert file_sha(src) == row[hash_key], f'File nguồn bị đổi: {src}'
        if src != dst:
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.is_file() or file_sha(dst) != row[hash_key]:
                # Input đã bung: đọc/ghi thường để tránh sendfile trên mount.
                with src.open('rb') as read, dst.open('wb') as write:
                    shutil.copyfileobj(read, write, length=1024*1024)
            assert file_sha(dst) == row[hash_key], dst
        # Archive: file đã trên ổ tạm và vừa được hash; không chép/hash lần hai.
    counts = Counter()
    for line in (DATA_DIR/row['label_path']).read_text(encoding='utf-8').splitlines():
        values = list(map(float, line.split()))
        assert len(values)==5 and all(math.isfinite(v) for v in values)
        cls,x,y,w,h = values
        assert cls.is_integer() and 0<=cls<8 and 0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1
        counts[int(cls)] += 1
    assert sum(counts.values()) == int(row['boxes']) and counts
    return counts

class_counts = Counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    for i, counts in enumerate(pool.map(stage_record, train_val), 1):
        class_counts.update(counts)
        if i % 2000 == 0 or i == len(train_val):
            print('Stage/hash/labels:', i, '/', len(train_val), flush=True)
assert sum(class_counts.values()) == 41631
assert not (DATA_DIR/'images/test').exists() and not (DATA_DIR/'labels/test').exists()
actual_files = {p.relative_to(DATA_DIR).as_posix() for folder in ['images','labels'] for p in (DATA_DIR/folder).rglob('*') if p.is_file() and not p.name.endswith('.cache')}
assert actual_files == {r[k] for r in train_val for k in ('image_path','label_path')}, 'Có file dư trong scratch; dùng session mới.'
DATA_YAML = WORK/'training_data.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path':str(DATA_DIR.resolve()), 'train':'images/train', 'val':'images/val', 'nc':8, 'names':dict(enumerate(NAMES))},sort_keys=False),encoding='utf-8')
resolved = check_det_dataset(str(DATA_YAML),autodownload=False)
assert not resolved.get('test')
for split in ['train','val']:
    assert Path(resolved[split]) == (DATA_DIR/'images'/split).resolve()
OUT = Path(OUTPUT_ROOT); OUT.mkdir(parents=True, exist_ok=True)
PREPARED = {'dataset_manifest_sha256':MANIFEST_SHA256,'checkpoint_sha256':BEST_SHA256,'all_input_images':EXPECTED_IMAGES,'training_images':len(train_val),'training_boxes':41631,'classes':NAMES,'class_id_7':'cups+bottles by user request','resume':False,'cls_remap':False}
(OUT/'prepared.json').write_text(json.dumps(PREPARED,indent=2),encoding='utf-8')
print('PREPARED: 12.000 train + 2.500 val; 500 test giữ riêng. Chưa train.')
print('Box train+val:', {NAMES[i]:class_counts[i] for i in range(8)})

In [ ]:
"""Small, portable integrity helpers for the Project8 experiment campaign."""
import hashlib
import json
import math
import os
import zipfile
from pathlib import Path


def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def select_epoch(rows, expected_epochs):
    """CSV is one based; reject incomplete/nonfinite history before choosing AP50."""
    if expected_epochs < 1 or len(rows) != expected_epochs:
        raise ValueError('Incomplete epoch history')
    parsed = []
    for index, row in enumerate(rows, 1):
        epoch = float(row['epoch'])
        ap50 = float(row['metrics/mAP50(B)'])
        ap = float(row['metrics/mAP50-95(B)'])
        if not all(math.isfinite(value) for value in (epoch, ap50, ap)):
            raise ValueError('Nonfinite epoch metric')
        if epoch != index or not 0 <= ap50 <= 1 or not 0 <= ap <= 1:
            raise ValueError('Invalid epoch sequence or AP range')
        parsed.append((ap50, ap, -index))
    return -max(parsed)[2]


def rank_results(results):
    """Only rank measured results using identical ground truth and protocol."""
    if not results:
        return []
    first = next(iter(results.values()))
    identity = (first['manifest_sha256'], first['protocol'])
    for record in results.values():
        if (record['manifest_sha256'], record['protocol']) != identity:
            raise ValueError('Cannot rank different datasets or protocols together')
        value = float(record['mAP50'])
        if not math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError('Invalid AP50')
    return sorted(results, key=lambda name: (-results[name]['mAP50'], name))


def verified_bundle(root, sources, destination):
    """Atomically publish a ZIP after CRC and every payload hash are verified."""
    root = Path(root).resolve()
    destination = Path(destination).resolve()
    payload = {}
    for source in sources:
        source = Path(source)
        if source.is_symlink():
            raise ValueError('Symlinks are not campaign payload')
        path = source.resolve()
        if not path.is_relative_to(root) or path == destination:
            raise ValueError('Payload outside run directory or recursive archive')
        name = path.relative_to(root).as_posix()
        if name == 'FILES.json' or name in payload:
            raise ValueError('Duplicate or reserved payload name')
        if not path.is_file():
            raise ValueError('Missing payload')
        payload[name] = path
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + '.partial')
    manifest = {name: {'sha256': file_sha(path), 'bytes': path.stat().st_size}
                for name, path in payload.items()}
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name, path in payload.items():
            archive.write(path, name)
        archive.writestr('FILES.json', json.dumps(manifest, indent=2))
    with zipfile.ZipFile(temporary) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt archive')
        for name, metadata in manifest.items():
            digest = hashlib.sha256()
            total = 0
            with archive.open(name) as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b''):
                    total += len(block)
                    digest.update(block)
            if digest.hexdigest() != metadata['sha256'] or total != metadata['bytes']:
                raise ValueError('Payload changed while building archive')
    os.replace(temporary, destination)
    return file_sha(destination)


"""Run after the four verified preflight cells of the v05 Kaggle notebook.

This source is joined with src/classroom_ai/experiments/experiment_campaign.py by the local
notebook builder. User authorized exploration; unchanged, unreviewed v05 GT.
"""
import csv
import datetime
import gc
import json
import shutil
import subprocess
import time
import traceback
from pathlib import Path

import torch
import ultralytics
import yaml
from ultralytics import YOLO
from ultralytics.cfg import get_cfg
from ultralytics.models.yolo.detect import DetectionValidator
from ultralytics.utils.nms import TorchNMS

assert ultralytics.__version__ == '8.4.165'
assert torch.cuda.device_count() >= 2
assert file_sha(initial) == BEST_SHA256
assert PREPARED['dataset_manifest_sha256'] == MANIFEST_SHA256
assert PREPARED['training_images'] == 14500
assert PREPARED['training_boxes'] == 41631
assert OUT.name == 'BTL-AI-Camera-v05-campaign-20261010'
assert len([r for r in rows if r['split'] == 'val']) == 2500
assert not (DATA_DIR / 'images/test').exists()

class CampaignNMSValidator(DetectionValidator):
    def postprocess(self, preds):
        items = super().postprocess(preds)
        assert self.end2end and self.args.nms is False
        for item in items:
            selected = []
            for cls in item['cls'].unique():
                ids = (item['cls'] == cls).nonzero(as_tuple=False).flatten()
                keep = TorchNMS.nms(item['bboxes'][ids], item['conf'][ids], .6)
                selected.append(ids[keep])
            if selected:
                keep = torch.cat(selected)
                keep = keep[item['conf'][keep].argsort(descending=True)]
                for key in ['bboxes', 'conf', 'cls', 'extra']:
                    item[key] = item[key][keep]
        return items

stamp = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).strftime('%Y%m%d_%H%M%S')
campaign_dir = OUT / ('batch_' + stamp)
campaign_dir.mkdir(exist_ok=False)
protocol = dict(data=str(DATA_YAML), split='val', imgsz=1280, batch=8,
                device=0, conf=.001, iou=.7, nms=False, max_det=300,
                augment=False, rect=True, quantize=16, workers=2, plots=True)
state = dict(status='running', target_mAP50=.80, parent_sha256=BEST_SHA256,
             manifest_sha256=MANIFEST_SHA256, protocol=protocol,
             full_visual_review_complete=bool(TRAIN_GATE),
             selection_note='Epoch, configurations and NMS selected on validation; not independent test.',
             started_at=stamp, runs={}, errors={}, archives={})

def write_state():
    temporary = campaign_dir / 'comparison.json.partial'
    temporary.write_text(json.dumps(state, indent=2), encoding='utf-8')
    temporary.replace(campaign_dir / 'comparison.json')

def preserve(label):
    trial_label = label.removesuffix('_trained').removesuffix('_complete').removesuffix('_error')
    selected_paths = {Path(r['checkpoint']) for r in state['runs'].values() if r.get('checkpoint')}
    sources = []
    for p in sorted(campaign_dir.rglob('*')):
        if not p.is_file() or p.suffix not in {'.pt', '.csv', '.yaml', '.json', '.jpg', '.png', '.py', '.txt'}:
            continue
        relative = p.relative_to(campaign_dir)
        include = len(relative.parts) == 1
        if relative.parts[0] == 'train':
            include = relative.parts[1] == trial_label if label != 'FINAL' else (p.suffix != '.pt' or p in selected_paths)
        if relative.parts[0] == 'evaluation':
            include = relative.parts[1].startswith(trial_label + '_') or label == 'FINAL'
        if include and p.name != 'FILES.json':
            sources.append(p)
    archive = OUT / ('campaign_' + stamp + '_' + label + '.zip')
    digest = verified_bundle(campaign_dir, sources, archive)
    state['archives'][label] = dict(path=str(archive), sha256=digest, bytes=archive.stat().st_size)
    write_state()
    print('VERIFIED_ARCHIVE', archive, digest, flush=True)

def evaluate(checkpoint, label):
    result = {}
    checkpoint_hash = file_sha(checkpoint)
    for mode in ['native', 'extra_nms_06']:
        model = YOLO(str(checkpoint))
        assert model.task == 'detect' and [model.names[i] for i in range(8)] == NAMES
        kwargs = dict(protocol, project=str(campaign_dir / 'evaluation'),
                      name=label + '_' + mode, exist_ok=False)
        if mode == 'extra_nms_06':
            kwargs['validator'] = CampaignNMSValidator
        metrics = model.val(**kwargs)
        assert model.model.end2end
        record = dict(checkpoint=str(checkpoint), checkpoint_sha256=checkpoint_hash,
                      manifest_sha256=MANIFEST_SHA256, protocol=protocol,
                      postprocessing=mode, extra_nms_iou=.6 if mode != 'native' else None,
                      mAP50=float(metrics.box.map50), mAP50_95=float(metrics.box.map),
                      precision=float(metrics.box.mp), recall=float(metrics.box.mr),
                      speed=metrics.speed, per_class={})
        for j, cls in enumerate(metrics.box.ap_class_index):
            record['per_class'][NAMES[int(cls)]] = dict(AP50=float(metrics.box.ap50[j]),
                                                      AP50_95=float(metrics.box.ap[j]))
        result[mode] = record
        (campaign_dir / (label + '_' + mode + '.json')).write_text(json.dumps(record, indent=2))
        print('CAMPAIGN_METRIC', label, mode, json.dumps(record), flush=True)
        del model, metrics
        gc.collect()
        torch.cuda.empty_cache()
    return result

base = dict(model=str(initial), data=str(DATA_YAML), project=str(campaign_dir / 'train'),
            device='0,1', workers=2, cache=False, fraction=1., patience=0,
            imgsz=1280, batch=32, optimizer='MuSGD', lr0=.0005, lrf=1.,
            cos_lr=False, warmup_epochs=.25, warmup_bias_lr=0., warmup_momentum=.8,
            momentum=.937, weight_decay=.0005, seed=42, deterministic=True,
            flipud=0., degrees=0., shear=0., perspective=0., bgr=0., mixup=0., cutmix=0.,
            close_mosaic=0, multi_scale=False, save=True, save_period=1,
            resume=False, cls_remap=False, val=True, nms=False, max_det=300,
            conf=.001, plots=True, exist_ok=False, **LIGHT_AUG)
trials = [
    dict(name='R01_recover_L1A1', epochs=2),
    dict(name='R02_light_longer', epochs=8),
    dict(name='R03_low_lr_base', epochs=4, lr0=.0001, **BASE_AUG),
    dict(name='R04_class_weight_025', epochs=4, cls_pw=.25),
    dict(name='R05_class_weight_05', epochs=4, cls_pw=.5),
    dict(name='R06_no_mosaic', epochs=4, mosaic=0., scale=.1, translate=.025),
    dict(name='R07_AdamW', epochs=4, optimizer='AdamW', lr0=.0001),
    dict(name='R08_freeze_backbone', epochs=3, freeze=10),
    dict(name='R09_imgsz1536', epochs=3, imgsz=1536, batch=16),
    dict(name='R10_mixup_cutmix', epochs=3, mixup=.1, cutmix=.1, **BASE_AUG),
    dict(name='R11_box_cls_loss', epochs=3, box=9., cls=.7),
    dict(name='R12_yolo26s', epochs=10, model='yolo26s.pt', batch=8,
         lr0=.001, warmup_epochs=1., lrf=.1, cos_lr=True, **BASE_AUG),
]
for trial in trials:
    get_cfg(overrides=dict(base, **trial))
state['planned_trials'] = trials
(campaign_dir / 'prepared.json').write_text(json.dumps(PREPARED, indent=2))
(campaign_dir / 'training_data.yaml').write_text(DATA_YAML.read_text())
if 'CAMPAIGN_SOURCE' in globals():
    (campaign_dir / 'executed_source.py').write_text(CAMPAIGN_SOURCE)
write_state()
print('CAMPAIGN_PREFLIGHT_PASS', campaign_dir, 'epochs_total=', sum(t['epochs'] for t in trials), flush=True)
state['runs']['BASELINE'] = dict(status='complete', measurements=evaluate(initial, 'BASELINE'))
write_state()
preserve('BASELINE')
for trial in trials:
    label = trial['name']
    config = dict(base, **trial)
    if shutil.disk_usage(OUT).free < 2 * 1024**3:
        state['status'] = 'stopped_disk_space'
        write_state()
        break
    config_path = campaign_dir / (label + '_config.yaml')
    config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    record = dict(status='training', config=config, parent_sha256=file_sha(initial)
                  if config['model'] == str(initial) else None,
                  initialization=config['model'], start=time.time())
    state['runs'][label] = record
    write_state()
    try:
        print('CAMPAIGN_TRAIN_START', label, json.dumps(config), flush=True)
        subprocess.run([shutil.which('yolo'), 'detect', 'train', f'cfg={config_path}'],
                       check=True, cwd=str(campaign_dir))
        run_dir = campaign_dir / 'train' / label
        with (run_dir / 'results.csv').open(newline='') as stream:
            history = [{k.strip(): v.strip() for k, v in r.items()} for r in csv.DictReader(stream)]
        epoch = select_epoch(history, config['epochs'])
        selected = run_dir / 'weights' / f'epoch{epoch-1}.pt'
        assert selected.is_file()
        record.update(status='trained', epoch_metrics=history, selected_epoch=epoch,
                      checkpoint=str(selected), checkpoint_sha256=file_sha(selected))
        write_state()
        preserve(label + '_trained')
        record['measurements'] = evaluate(selected, label)
        record.update(status='complete', elapsed_seconds=time.time() - record['start'])
        write_state()
        preserve(label + '_complete')
        print('CAMPAIGN_TRIAL_COMPLETE', label, flush=True)
    except Exception as exc:
        record['status'] = 'error'
        state['errors'][label] = dict(type=type(exc).__name__, message=str(exc), traceback=traceback.format_exc())
        write_state()
        preserve(label + '_error')
        print('CAMPAIGN_TRIAL_ERROR', label, traceback.format_exc(), flush=True)
        gc.collect()
        torch.cuda.empty_cache()
else:
    state['status'] = 'batch_complete'
write_state()
preserve('FINAL')
print('CAMPAIGN_BATCH_DONE', state['status'], campaign_dir, flush=True)
